<a href="https://colab.research.google.com/github/priyanshusingh303/StrokeAi/blob/main/handwritten_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q opencv-python kaggle gradio scikit-learn
import os, numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix

tf.keras.utils.set_random_seed(42)


2.20.0 []


Download EMNIST from Kaggle

In [3]:
from google.colab import files
files.upload()   # select kaggle.json

!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d crawford/emnist -p /content/emnist --unzip
!ls /content/emnist

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/crawford/emnist
License(s): CC0-1.0
100% 1.24G/1.24G [00:10<00:00, 122MB/s]

emnist-balanced-mapping.txt  emnist-digits-test.csv
emnist-balanced-test.csv     emnist-digits-train.csv
emnist-balanced-train.csv    emnist-letters-mapping.txt
emnist-byclass-mapping.txt   emnist-letters-test.csv
emnist-byclass-test.csv      emnist-letters-train.csv
emnist-byclass-train.csv     emnist-mnist-mapping.txt
emnist-bymerge-mapping.txt   emnist-mnist-test.csv
emnist-bymerge-test.csv      emnist-mnist-train.csv
emnist-bymerge-train.csv     emnist_source_files
emnist-digits-mapping.txt


Load both datasets and build the input pipeline

In [5]:
#EMNIST ByClass (62 classes)
def load_emnist(path):
    v = pd.read_csv(path, header=None, dtype=np.uint8).values
    y = v[:, 0].astype('int64')
    X = v[:, 1:].reshape(-1, 28, 28, 1)
    return np.transpose(X, (0, 2, 1, 3)).copy(), y     # fix EMNIST rotation

X_em, y_em = load_emnist('/content/emnist/emnist-byclass-train.csv')
X_emt, y_emt = load_emnist('/content/emnist/emnist-byclass-test.csv')

mapping = pd.read_csv('/content/emnist/emnist-byclass-mapping.txt', sep=' ', header=None)
EM_LABELS = [chr(c) for c in mapping[1]]     # indices 0-9 are digits '0'-'9'

p = np.random.RandomState(42).permutation(len(X_em))
X_em, y_em = X_em[p], y_em[p]
X_em_val, y_em_val = X_em[:40000], y_em[:40000]
X_em_tr,  y_em_tr  = X_em[40000:], y_em[40000:]

#MNIST (digits)
(Xm, ym), (Xm_te, ym_te) = tf.keras.datasets.mnist.load_data()
Xm, Xm_te = Xm[..., None], Xm_te[..., None]
DIGITS = list("0123456789")
p = np.random.RandomState(42).permutation(len(Xm))
Xm, ym = Xm[p], ym[p]
Xm_val, ym_val = Xm[:5000], ym[:5000]
Xm_tr,  ym_tr  = Xm[5000:], ym[5000:]

print("EMNIST:", X_em_tr.shape, len(EM_LABELS), "| MNIST:", Xm_tr.shape)

# ---------- tf.data pipeline with augmentation ----------
aug = tf.keras.Sequential([
    layers.RandomRotation(0.06),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.08, 0.08),
])

def make_ds(X, y, train=False, bs=256):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if train: ds = ds.shuffle(100_000)
    ds = ds.batch(bs).map(lambda a, b: (tf.cast(a, tf.float32) / 255.0, b),
                          num_parallel_calls=tf.data.AUTOTUNE)
    if train:
        ds = ds.map(lambda a, b: (aug(a, training=True), b),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
EMNIST: (657932, 28, 28, 1) 62 | MNIST: (55000, 28, 28, 1)
